# AAI-520 Final Team Project: Multi-Agent Financial Analysis System

## Prompt Chaining and Data Research


This section will develop the research planning and prompt chaining components of the Investment Research Agent. The workflow will take a stock symbol as input and use Gemini to create a research plan that identifies the main areas needed for financial analysis. Recent financial news will then be retrieved from Yahoo Finance and processed through five prompt chaining stages: **ingest, preprocess, classify, extract, and summarize**.

Gemini will be used throughout the workflow to dynamically classify financial news, extract important financial events and findings, and generate a concise summary of the results. The final output will combine the research plan with structured financial news data so that the information can be passed to the routing and specialist agents in the larger multi-agent financial analysis system.

## Install Required Library

Yahoo Finance will be used to retrieve financial market and news data for the selected stock symbol.

In [1]:
# Install the required libraries
!pip install -q yfinance google-genai

## Import Libraries

The required Python libraries will be imported for retrieving and processing the financial data.

In [2]:
# Import the libraries needed for the financial analysis
import yfinance as yf
import pandas as pd
from google import genai

In [3]:
# Connect to Gemini using the API key stored in Colab Secrets
from google.colab import userdata

gemini_api_key = userdata.get("GEMINI_API_KEY")
client = genai.Client(
    api_key=gemini_api_key,
    http_options={"timeout": 30000}
)

print("Gemini client connected successfully.")

Gemini client connected successfully.


In [4]:
# Create a reusable function for sending prompts to Gemini
import time

last_llm_call = 0

def call_llm(prompt):
    global last_llm_call

    # Space Gemini requests to stay within the free-tier rate limit
    elapsed = time.time() - last_llm_call
    if elapsed < 15:
        time.sleep(15 - elapsed)

    for attempt in range(3):
        try:
            response = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=prompt
            )

            last_llm_call = time.time()
            return response.text

        except Exception:
            if attempt < 2:
                time.sleep(15)
            else:
                raise

## Select a Stock Symbol

A stock symbol will be provided as the input for the Investment Research Agent.

In [5]:
# Select a stock symbol for the analysis
ticker_symbol = "AAPL"

# Create the Yahoo Finance ticker object
stock = yf.Ticker(ticker_symbol)

print("Stock Symbol:", ticker_symbol)

Stock Symbol: AAPL


## Research Planning

The Investment Research Agent will create a research plan for the selected stock symbol. The plan will define the steps needed to collect and analyze financial news before producing a final summary.

In [6]:
# Create a research plan for the selected stock using Gemini
def create_research_plan(symbol):
    prompt = f"""
    You are a financial research agent analyzing {symbol}.

    Create a concise research plan for analyzing the stock using
    available financial market data and recent financial news.

    Determine the research steps that should be completed before
    producing the final financial analysis.

    Return the plan as a numbered list.
    """

    return call_llm(prompt).strip()


# Generate the research plan
research_plan = create_research_plan(ticker_symbol)

# Display the research plan
print(f"Research Plan for {ticker_symbol}\n")
print(research_plan)

Research Plan for AAPL

Here is a concise research plan for analyzing AAPL:

1.  **Gather Core Financial Data:** Collect recent and historical financial statements (Income Statement, Balance Sheet, Cash Flow), key financial ratios (P/E, EV/EBITDA, ROE, Debt/Equity, FCF Yield), and historical stock performance data (price, volume) for AAPL and relevant peers.
2.  **Review Latest Earnings Reports & Guidance:** Analyze the most recent quarterly and annual earnings reports, investor calls, and management guidance to understand current performance, segment breakdown, and forward-looking outlook.
3.  **Monitor Recent News & Market Developments:** Compile and synthesize recent news regarding AAPL's product launches (e.g., Vision Pro updates), supply chain health, regulatory challenges (e.g., antitrust cases), competitive actions, strategic partnerships, and overall macroeconomic trends impacting consumer spending and tech.
4.  **Conduct Quantitative Valuation & Performance Analysis:**
    *  

## Prompt Chaining

### Step 1: Ingest News

Recent financial news will be retrieved from Yahoo Finance for the selected stock symbol. The retrieved news will provide the input for the remaining steps of the prompt chaining workflow.

In [7]:
# Ingest recent news for the selected stock
def ingest_news(symbol):
    stock = yf.Ticker(symbol)
    news_items = stock.news

    # Use Yahoo Finance search as a fallback if ticker news is empty
    if not news_items:
        search = yf.Search(symbol, news_count=10)
        news_items = search.news

    return news_items


# Retrieve recent news
news = ingest_news(ticker_symbol)

# Display the number of retrieved news items
print("Number of news items:", len(news))

# Display the first news item
news[0] if news else "No news found"

Number of news items: 9


{'uuid': 'e09e750c-1a63-3c36-b829-929a5475c34d',
 'title': 'Micron Technology vs. Qualcomm: What Revenue Trends Tell Investors About These Tech Companies',
 'publisher': 'Motley Fool',
 'link': 'https://finance.yahoo.com/markets/stocks/articles/micron-technology-vs-qualcomm-revenue-054012853.html',
 'providerPublishTime': 1791092412,
 'type': 'STORY',
 'thumbnail': {'resolutions': [{'url': 'https://media.zenfs.com/en/motleyfool.com/1f0880597c2de0fafaf2d488c1b05838.png',
    'width': 1200,
    'height': 900,
    'tag': 'original'},
   {'url': 'https://s.yimg.com/lo/mysterio/api/AD95039629CBECECD3916B3B1310DCB2150BA2143CDBEEDEA8008FEB7D0B467B/subgraphmysterio/resizefill_w140_h140/https:%2F%2Fmedia.zenfs.com%2Fen%2Fmotleyfool.com%2F1f0880597c2de0fafaf2d488c1b05838.png',
    'width': 140,
    'height': 140,
    'tag': '140x140'}]},
 'relatedTickers': ['QCOM', 'MU', 'GM', 'F', 'BMW.DE', 'AAPL']}

### Step 2: Preprocess

The retrieved Yahoo Finance news contains additional metadata that is not needed for the analysis. The news will be preprocessed by keeping the title, summary, publication date, provider, and article URL for each item.

In [8]:
# Preprocess the retrieved news items
def preprocess_news(news_items):
    processed_news = []

    for item in news_items:
        # Handle the standard ticker news format
        if "content" in item:
            content = item.get("content", {})

            processed_news.append({
                "title": content.get("title", ""),
                "summary": content.get("summary", ""),
                "pub_date": content.get("pubDate", ""),
                "provider": content.get("provider", {}).get("displayName", ""),
                "url": content.get("canonicalUrl", {}).get("url", "")
            })

        # Handle the Yahoo Finance search fallback format
        else:
            processed_news.append({
                "title": item.get("title", ""),
                "summary": item.get("summary", ""),
                "pub_date": item.get("providerPublishTime", ""),
                "provider": item.get("publisher", ""),
                "url": item.get("link", "")
            })

    return processed_news


# Preprocess the Yahoo Finance news
processed_news = preprocess_news(news)

# Convert the processed news into a DataFrame
news_df = pd.DataFrame(processed_news)

# Convert Unix timestamps to readable publication dates
news_df["pub_date"] = pd.to_datetime(
    news_df["pub_date"],
    unit="s",
    errors="coerce"
)

# Display the processed news
news_df

,title,summary,pub_date,provider,url
0,Micron Technology vs. Qualcomm: What Revenue T...,,2026-10-04 05:40:12,Motley Fool,https://finance.yahoo.com/markets/stocks/artic...
1,ASML Holding vs. Qualcomm: What Revenue Trends...,,2026-10-03 22:52:32,Motley Fool,https://finance.yahoo.com/markets/stocks/artic...
2,"Dear Apple Stock Fans, Mark Your Calendars for...",,2026-10-03 18:30:02,Barchart,https://finance.yahoo.com/markets/stocks/artic...
3,This Stock Posted the Best Performance in the ...,,2026-10-03 13:30:01,Motley Fool,https://finance.yahoo.com/markets/stocks/artic...
4,"Mag 7 Voices: Huang Defends AI Spending, Picha...",,2026-10-03 02:01:35,Stocktwits,https://finance.yahoo.com/technology/ai/articl...
5,Apple Says Some AT&T iPhone 18 Pro Max Users M...,,2026-10-02 23:03:33,MT Newswires,https://finance.yahoo.com/technology/articles/...
6,Apple Tightens macOS Full Disk Access,,2026-10-02 22:06:15,MT Newswires,https://finance.yahoo.com/technology/ai/articl...
7,Apple iPhone 18 Pro Max AT&T Glitch Requires D...,,2026-10-02 22:01:28,Bloomberg,https://finance.yahoo.com/technology/articles/...
8,Apple (AAPL) Exceeds Market Returns: Some Fact...,,2026-10-02 20:45:04,Zacks,https://finance.yahoo.com/markets/stocks/artic...


### Step 3: Classify

The preprocessed news will be classified into financial categories using Gemini. Each article title and summary will be analyzed, and an appropriate category will be determined dynamically based on the article's content.

In [9]:
# Classify all news items dynamically using Gemini
def classify_news(data):
    articles = ""

    for index, row in data.iterrows():
        articles += (
            f"\nArticle {index}:\n"
            f"Title: {row['title']}\n"
            f"Summary: {row['summary']}\n"
        )

    prompt = f"""
    Analyze each financial news article below and assign one concise,
    relevant financial category based on the article's actual content.

    Determine the category dynamically. Do not use a predefined list
    of categories.

    Use a short category name that clearly describes the main financial
    topic of the article.

    Return exactly one result per article using this format:
    Article 0: Category
    Article 1: Category

    Articles:
    {articles}
    """

    response = call_llm(prompt)

    categories = {}

    for line in response.splitlines():
        if ":" in line and line.strip().startswith("Article"):
            article, category = line.split(":", 1)

            try:
                index = int(article.replace("Article", "").strip())
                categories[index] = category.strip()
            except ValueError:
                continue

    return categories


# Classify the news articles with Gemini
categories = classify_news(news_df)

# Add the dynamically generated categories to the DataFrame
news_df["category"] = news_df.index.map(categories)

# Display the classification results
news_df[["title", "category"]]

,title,category
0,Micron Technology vs. Qualcomm: What Revenue T...,Revenue Trend Analysis
1,ASML Holding vs. Qualcomm: What Revenue Trends...,Revenue Trend Analysis
2,"Dear Apple Stock Fans, Mark Your Calendars for...",Stock Event Alert
3,This Stock Posted the Best Performance in the ...,Stock Performance Review
4,"Mag 7 Voices: Huang Defends AI Spending, Picha...",Tech Company Updates
5,Apple Says Some AT&T iPhone 18 Pro Max Users M...,Product Defect Report
6,Apple Tightens macOS Full Disk Access,Software Security Update
7,Apple iPhone 18 Pro Max AT&T Glitch Requires D...,Product Defect Report
8,Apple (AAPL) Exceeds Market Returns: Some Fact...,Stock Performance Analysis


### Step 4: Extract

Gemini will extract the key financial information from the classified news. The extraction will identify the main financial event or finding from each article so that the most relevant information can be passed to the summarization stage.

In [10]:
# Extract key financial information using Gemini
def extract_information(data):
    articles = ""

    for index, row in data.iterrows():
        articles += (
            f"\nArticle {index}\n"
            f"Title: {row['title']}\n"
            f"Category: {row['category']}\n"
            f"Summary: {row['summary']}\n"
        )

    prompt = f"""
    Extract the main financial event or finding from each article below.

    Keep each extraction concise and based only on the provided information.

    Return one result per line using this exact format:
    Article 0: extracted information
    Article 1: extracted information

    Articles:
    {articles}
    """

    response = call_llm(prompt)

    extractions = {}

    for line in response.splitlines():
        if ":" in line:
            article, information = line.split(":", 1)

            try:
                index = int(article.replace("Article", "").strip())
                extractions[index] = information.strip()
            except ValueError:
                continue

    extracted_data = data[
        ["title", "category", "pub_date", "provider"]
    ].copy()

    extracted_data["key_information"] = extracted_data.index.map(extractions)

    return extracted_data


# Extract the important financial information
extracted_news = extract_information(news_df)

# Display the extracted information
extracted_news

,title,category,pub_date,provider,key_information
0,Micron Technology vs. Qualcomm: What Revenue T...,Revenue Trend Analysis,2026-10-04 05:40:12,Motley Fool,Revenue trend analysis for Micron Technology a...
1,ASML Holding vs. Qualcomm: What Revenue Trends...,Revenue Trend Analysis,2026-10-03 22:52:32,Motley Fool,Revenue trend analysis for ASML Holding and Qu...
2,"Dear Apple Stock Fans, Mark Your Calendars for...",Stock Event Alert,2026-10-03 18:30:02,Barchart,Apple stock event alert for October 13.
3,This Stock Posted the Best Performance in the ...,Stock Performance Review,2026-10-03 13:30:01,Motley Fool,"A ""Magnificent Seven"" stock posted the best pe..."
4,"Mag 7 Voices: Huang Defends AI Spending, Picha...",Tech Company Updates,2026-10-03 02:01:35,Stocktwits,Huang defends AI spending.
5,Apple Says Some AT&T iPhone 18 Pro Max Users M...,Product Defect Report,2026-10-02 23:03:33,MT Newswires,Apple requires replacement of some AT&T iPhone...
6,Apple Tightens macOS Full Disk Access,Software Security Update,2026-10-02 22:06:15,MT Newswires,Apple tightens macOS Full Disk Access security.
7,Apple iPhone 18 Pro Max AT&T Glitch Requires D...,Product Defect Report,2026-10-02 22:01:28,Bloomberg,Apple iPhone 18 Pro Max AT&T glitch requires d...
8,Apple (AAPL) Exceeds Market Returns: Some Fact...,Stock Performance Analysis,2026-10-02 20:45:04,Zacks,Apple (AAPL) exceeds market returns.


### Step 5: Summarize

Gemini will summarize the extracted financial information into a concise overview for the selected stock. The summary will use the classified and extracted findings from the previous stages of the prompt chaining workflow.

In [11]:
# Summarize the extracted financial information using Gemini
def summarize_news(data, symbol):
    findings = ""

    for _, row in data.iterrows():
        findings += (
            f"\nCategory: {row['category']}\n"
            f"Finding: {row['key_information']}\n"
        )

    prompt = f"""
    Summarize the following financial news findings for {symbol}.

    Provide a concise financial news overview based only on the
    information provided below. Highlight the most important company
    and market developments. Do not add outside information.

    Findings:
    {findings}
    """

    return call_llm(prompt).strip()


# Generate the Gemini financial news summary
news_summary = summarize_news(extracted_news, ticker_symbol)

# Display the final summary
print(news_summary)

Based solely on the provided information, Apple (AAPL) has exceeded market returns. An Apple stock event alert is noted for October 13. On the product and software front, Apple requires replacements for some AT&T iPhone 18 Pro Max devices due to a bug or glitch, and the company has tightened macOS Full Disk Access security.


## Complete Prompt Chaining Workflow

The five prompt chaining steps will now be connected into one workflow. The workflow will take a stock symbol as input, retrieve its recent news, process the information, and return the final results.

In [12]:
# Connect all five prompt chaining steps into one workflow
def run_prompt_chain(symbol):
    # Step 1: Ingest news
    news_items = ingest_news(symbol)

    # Step 2: Preprocess news
    processed = preprocess_news(news_items)
    data = pd.DataFrame(processed)

    # Convert Unix timestamps to readable publication dates
    data["pub_date"] = pd.to_datetime(
     data["pub_date"],
     unit="s",
     errors="coerce"
    )

    # Step 3: Classify news using Gemini
    categories = classify_news(data)
    data["category"] = data.index.map(categories)

    # Step 4: Extract financial information using Gemini
    extracted = extract_information(data)

    # Step 5: Summarize the findings using Gemini
    summary = summarize_news(extracted, symbol)

    return extracted, summary


# Run the complete workflow for the selected stock
final_news, final_summary = run_prompt_chain(ticker_symbol)

print(f"Prompt Chaining Results for {ticker_symbol}\n")
print(final_summary)

Prompt Chaining Results for AAPL

Based on the findings provided:

Apple (AAPL) stock has exceeded overall market returns. The company faces a significant product issue, requiring replacement of some AT&T iPhone 18 Pro Max devices due to a service-loss bug or glitch. In other product news, Apple has tightened macOS Full Disk Access for enhanced security. Investors should note an upcoming significant date for Apple stock on October 13.


### Prompt Chaining Workflow Output

The completed prompt chaining workflow will provide structured financial news data for use by the other agents in the multi-agent system. The output will include the selected stock symbol, research plan, classified and extracted news information, and the Gemini-generated financial news summary.

In [13]:
# Create the final structured output for the prompt chaining workflow
prompt_chain_output = {
    "symbol": ticker_symbol,
    "research_plan": research_plan,
    "processed_news": final_news.to_dict(orient="records"),
    "summary": final_summary
}

# Display the structure of the final output
print("Stock Symbol:", prompt_chain_output["symbol"])
print("Processed News Articles:", len(prompt_chain_output["processed_news"]))
print("\nResearch Plan:")
print(prompt_chain_output["research_plan"])
print("\nFinal Summary:")
print(prompt_chain_output["summary"])

Stock Symbol: AAPL
Processed News Articles: 9

Research Plan:
Here is a concise research plan for analyzing AAPL:

1.  **Gather Core Financial Data:** Collect recent and historical financial statements (Income Statement, Balance Sheet, Cash Flow), key financial ratios (P/E, EV/EBITDA, ROE, Debt/Equity, FCF Yield), and historical stock performance data (price, volume) for AAPL and relevant peers.
2.  **Review Latest Earnings Reports & Guidance:** Analyze the most recent quarterly and annual earnings reports, investor calls, and management guidance to understand current performance, segment breakdown, and forward-looking outlook.
3.  **Monitor Recent News & Market Developments:** Compile and synthesize recent news regarding AAPL's product launches (e.g., Vision Pro updates), supply chain health, regulatory challenges (e.g., antitrust cases), competitive actions, strategic partnerships, and overall macroeconomic trends impacting consumer spending and tech.
4.  **Conduct Quantitative Valua

## Sample Processed News

A sample of the processed financial news will be displayed to demonstrate the structured output produced by the prompt chaining workflow. This output can be passed to the routing and specialist agents for further analysis.

In [14]:
# Display sample processed articles for integration with other agents
sample_articles = final_news.head(3)

sample_articles

,title,category,pub_date,provider,key_information
0,Micron Technology vs. Qualcomm: What Revenue T...,Tech Stock Analysis,2026-10-04 05:40:12,Motley Fool,Comparison of revenue trends between Micron Te...
1,ASML Holding vs. Qualcomm: What Revenue Trends...,Semiconductor Stock Analysis,2026-10-03 22:52:32,Motley Fool,Comparison of revenue trends between ASML Hold...
2,"Dear Apple Stock Fans, Mark Your Calendars for...",Corporate Event Alert,2026-10-03 18:30:02,Barchart,An upcoming significant date for Apple stock o...


## AI-Assisted Tools Disclosure

Generative AI tools were used to support the development and implementation of this assignment. Google Gemini (`gemini-2.5-flash`) is integrated directly into the financial analysis workflow through the Gemini API. Gemini is used to generate a research plan for the selected stock, dynamically classify financial news articles, extract key financial information, and summarize the extracted findings. The prompts and outputs are incorporated into the prompt chaining workflow and are demonstrated throughout the notebook.

ChatGPT was also used as a development support tool during the assignment. It assisted with reviewing code structure, troubleshooting implementation issues, improving code comments and documentation, and identifying potential improvements to the workflow. All generated suggestions were reviewed, tested, and modified as needed before being incorporated into the notebook.

Yahoo Finance (`yfinance`) is used as the financial data and news source. The final implementation, integration, testing, and interpretation of the workflow were completed and verified by the student. AI-generated responses were not treated as automatically correct; outputs were reviewed through execution and testing to confirm that the workflow performed as intended.

## References

Google. (2026). *Gemini API documentation*. Google AI for Developers. https://ai.google.dev/gemini-api/docs

OpenAI. (2026). *ChatGPT* [Large language model]. https://chatgpt.com/

yfinance. (2026). *yfinance documentation*. https://ranaroussi.github.io/yfinance/